In [ ]:
!git clone https://github.com/jburroni/IntroPPLs26.git

%cd IntroPPLs26/notebooks/Jun-29

# Activity 6 - MAP, RWMH, and HMC as controllers over the message interface

Activity 5 split model from inference: the stack machine emits `sample` and `observe`
messages, and a **controller** answers them. Likelihood weighting and SMC were two
controllers over one runtime. Here we add three more, and the new ingredient is gradients.

With a fixed set of continuous latents (**static support**), running the program is a
deterministic map from a trace `X` to its **potential energy** $U(X) = -\log\gamma(X)$. Make
the latents PyTorch tensors and the runtime is differentiable, so:

- **MAP** finds the mode by minimizing $U$ (a warm-up: it checks the machinery);
- **RWMH** samples with $U$ alone, and struggles on a correlated posterior;
- **HMC** uses $\nabla U$ to sample along the geometry, where RWMH crawled.

The only change to the Activity 5 evaluator is one line: rebind `normal` to a differentiable
distribution. You complete the controller that turns `sample` and `observe` messages into the
potential.

In [ ]:
import os, sys, math
# locate the repo's interpreters/ folder by walking up, so this runs from anywhere
_p = os.path.abspath(os.getcwd())
while _p != os.path.dirname(_p):
    if os.path.isdir(os.path.join(_p, "interpreters", "minippl")):
        sys.path.insert(0, os.path.join(_p, "interpreters")); break
    _p = os.path.dirname(_p)

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.distributions as D
from minippl import parse, parse_one, Symbol, PRIMITIVES, is_primitive

torch.set_default_dtype(torch.double)

## The Activity 5 runtime, unchanged

The same stack machine: `resume(m)` runs until the next probabilistic effect and returns a
message (`'sample'`, `'observe'`, or `'done'`) carrying the paused machine. Nothing here is
modified for differentiation. Addresses are structural (a syntactic path), so on a
fixed-structure program every run names the same latents: that is the static support MAP and
HMC need.

In [ ]:
class Closure:
    __slots__ = ('params', 'body', 'env')
    def __init__(self, params, body, env):
        self.params, self.body, self.env = params, body, env   # body plus its defining env

class M:
    # a suspendable execution; C + V + state is the continuation
    def __init__(self, C, V=None, env=None, rng=None, log_w=0.0):
        self.C = list(C)
        self.V = [] if V is None else list(V)
        self.env = {} if env is None else dict(env)
        self.rng = rng; self.log_w = log_w
    def fork(self, rng=None):                                   # duplicate a paused machine: copy the stacks
        return M(C=list(self.C), V=list(self.V), env=dict(self.env),
                 rng=self.rng if rng is None else rng, log_w=self.log_w)
    def send(self, value):
        self.V.append(value)

def _push_body(C, body, env, addr):
    seq = []
    for n, b in enumerate(body[:-1]):
        seq.append(('ev', b, env, addr + ('body', n))); seq.append(('discard',))   # earlier body exprs run for effect
    seq.append(('ev', body[-1], env, addr + ('body', len(body) - 1)))              # the last is the value
    for item in reversed(seq):
        C.append(item)

def resume(m):
    C, V = m.C, m.V
    while C:
        instr = C.pop(); t = instr[0]
        if t == 'ev':
            _, e, env, addr = instr
            if isinstance(e, Symbol):
                if e in env: V.append(env[e])
                elif is_primitive(e): V.append(PRIMITIVES[e])
                else: raise NameError(e)
            elif not isinstance(e, list):
                V.append(e)
            else:
                head = e[0]
                if head == 'let':
                    binds, body = e[1], e[2:]
                    if binds:
                        C.append(('letk', binds, 0, body, env, addr)); C.append(('ev', binds[1], env, addr + ('let', 0)))
                    else:
                        _push_body(C, body, env, addr)
                elif head == 'if':
                    _, test, then, els = e
                    C.append(('ifk', then, els, env, addr)); C.append(('ev', test, env, addr + ('test',)))
                elif head == 'fn':
                    _, params, *body = e
                    V.append(Closure(params, body, env))        # capture the defining env
                elif head == 'sample':
                    C.append(('samplek', addr)); C.append(('ev', e[1], env, addr + ('d',)))
                elif head == 'observe':
                    C.append(('observek', addr)); C.append(('ev', e[2], env, addr + ('v',))); C.append(('ev', e[1], env, addr + ('d',)))
                else:                                            # application: operator then args, each at its own address
                    C.append(('callk', len(e) - 1, addr))
                    for i in range(len(e) - 1, 0, -1): C.append(('ev', e[i], env, addr + (i - 1,)))
                    C.append(('ev', e[0], env, addr + ('fn',)))
        elif t == 'letk':
            _, binds, i, body, env, addr = instr
            env = dict(env); env[binds[2*i]] = V.pop()           # fresh env: no shared mutation, so fork is safe
            if 2*(i+1) < len(binds):
                C.append(('letk', binds, i+1, body, env, addr)); C.append(('ev', binds[2*(i+1)+1], env, addr + ('let', 2*(i+1))))
            else:
                _push_body(C, body, env, addr)
        elif t == 'ifk':
            _, then, els, env, addr = instr
            branch, tag = (then, 'then') if V.pop() else (els, 'else')   # the branch taken is part of the address
            C.append(('ev', branch, env, addr + (tag,)))
        elif t == 'discard':
            V.pop()
        elif t == 'callk':
            _, n, addr = instr
            args = [V.pop() for _ in range(n)][::-1]; f = V.pop()
            if isinstance(f, Closure):
                if len(args) != len(f.params):
                    raise TypeError(f"expected {len(f.params)} args, got {len(args)}")
                new_env = dict(f.env)
                for p, arg in zip(f.params, args): new_env[p] = arg
                _push_body(C, f.body, new_env, addr)
            else:
                V.append(f(*args))                               # primitive
        elif t == 'samplek':
            _, addr = instr
            d = V.pop()
            return ('sample', addr, d, m)                        # no draw: emit a message
        elif t == 'observek':
            _, addr = instr
            y = V.pop(); d = V.pop()
            return ('observe', addr, d, y, m)                    # no score: emit a message
    return ('done', V[-1], m)


## Making the runtime differentiable

`minippl`'s `Normal.log_prob` coerces its argument to `float`, which would strip a tensor's
gradient. We rebind `normal` to a differentiable `torch.distributions.Normal`. This is the
only change, and it is the book's "rebind the primitives to AD-compatible versions" (7.3):
the evaluator's arithmetic is operator overloading, so tensors already flow through it.

In [ ]:
def _normal(mu, sigma):
    mu = mu if torch.is_tensor(mu) else torch.tensor(float(mu))
    sigma = sigma if torch.is_tensor(sigma) else torch.tensor(float(sigma))
    return D.Normal(mu, sigma)

PRIMITIVES["normal"] = _normal     # rebind in place; the evaluator looks names up here

## The model and its exact posterior

Bayesian linear regression with two continuous latents, the slope `a` and the intercept `b`,
and a fixed structure (no stochastic control flow), so the trace has static support. Gaussian
prior and Gaussian likelihood make the posterior exactly Gaussian, with a closed form we
check against.

```clojure
(let [a (sample (normal 0 1))
      b (sample (normal 0 1))]
  (observe (normal (+ (* a x_n) b) 0.3) y_n)   ; for each point
  (vector a b))
```

In [ ]:
SIGMA = 0.3
gen = np.random.default_rng(0); N = 12
xs = np.linspace(-1.0, 4.0, N)
ys = 1.8 * xs - 0.7 + gen.normal(0, SIGMA, size=N)        # truth a=1.8, b=-0.7

src = "(let [a (sample (normal 0 1))\n      b (sample (normal 0 1))]\n"
for x, y in zip(xs, ys):
    src += f"  (observe (normal (+ (* a {float(x):.6f}) b) {SIGMA}) {float(y):.6f})\n"
src += "  (vector a b))"

class Model:
    # owns a program and the latent address set its static support fixes
    def __init__(self, src):
        self.program = parse_one(src)
        self.addrs = self._discover()                     # PASS 1: fix A = (a_1, ..., a_d)

    def machine(self):                                    # a fresh paused run of the program
        return M([("ev", self.program, {}, ())], env={}, rng=None)

    def _discover(self, sample_values=None):                                  # run once in discovery mode for the addresses
        m = self.machine(); addrs = []
        if sample_values is not None:
            it = iter(sample_values)
        else:
            def zero_sample(): 
                while True: yield torch.zeros(())
            it = zero_sample()
        while True:
        msg = resume(m)
        tag = msg[0]
        if tag == "done":
            # TODO: return the ordered list of sample addresses
            raise NotImplementedError
        elif tag == "sample":
            # msg = ("sample", address, distribution, machine)
            # TODO:
            #   1. unpack the message
            #   2. record the address
            #   3. send the next deterministic value into the machine
            raise NotImplementedError
        elif tag == "observe":
            # msg = ("observe", address, distribution, observed_value, machine)
            # TODO:
            #   1. unpack the message
            #   2. send the observed value back into the machine
            raise NotImplementedError

class StaticSupportError(Exception):
    pass

model = Model(src)

# closed-form Gaussian posterior: the MAP mode equals the mean
Phi = np.column_stack([xs, np.ones_like(xs)])
Sig = np.linalg.inv(np.eye(2) + Phi.T @ Phi / SIGMA**2)
MODE = Sig @ (Phi.T @ ys) / SIGMA**2
print(f"closed-form posterior mode (a,b) = ({MODE[0]:.4f}, {MODE[1]:.4f})")
print("latent addresses (static support):", model.addrs)

## When static support fails

The check is not decoration. A program whose latent set depends on a **value** leaves the
fragment, and the controller raises instead of silently returning a wrong density. Below a
second latent appears only when `a > 0`: discovery (run at `a = 0`) sees one latent, but an
evaluation at `a > 0` visits two.

In [ ]:
dyn = Model("(let [a (sample (normal 0 1))]"
                "  (if (> a 0.0) (sample (normal 0 1)) 0)"
                "  a)")


discovered = dyn._discover([0.0])           # a = 0  -> branch false -> one latent
visited    = dyn._discover([1.0, 0.0])      # a > 0  -> a second latent appears
print("discovered:", discovered)
print("visited at a>0:", visited)
try:
    if visited != discovered:
        raise StaticSupportError(f"expected {discovered}, but visited {visited}")
except StaticSupportError as e:
    print("rejected:", e)

## The potential controller, with a static-support check

Pass 1 (above) fixed the latent vector $X = (x_{a_1}, \ldots, x_{a_d})$. **Pass 2** evaluates
the potential, and makes static support *operational*: every evaluation must visit **exactly**
those addresses. At a `sample` site read $x_{a_i}$ and add $-\log p_i(x_{a_i})$ to $U$; at an
`observe` site add $-\log p_j(y_j\mid X)$. After the run, if the visited addresses differ from
`model.addrs`, the program has left the fragment MAP and HMC handle, and we raise rather than return
a wrong density.

**Complete the two branches.** The address bookkeeping and the check are given; you write the
scoring. Everything after depends on it, so the test cell fails until both branches are right.

In [ ]:
def potential(model, theta):
    # Pass 2: compute U(X) = -log gamma(X), requiring the run to visit exactly self.addrs.
    m = model.machine(); U = torch.zeros(()); visited = []
    while True:
        msg = resume(m); tag = msg[0]
        if tag == "done":
            if visited != model.addrs:                   # the dynamic static-support check
                raise StaticSupportError(f"expected sample addresses {model.addrs}, but visited {visited}")
            return U
        elif tag == "sample":
            _, a, d, m = msg
            visited.append(a)
            if a not in theta:                          # a latent we never discovered in pass 1
                raise StaticSupportError(f"left the static-support fragment at address {a}")
            # TODO: read the latent value theta[a], subtract its log-density under d from U
            # (use .sum() on log_prob, in case it is vector-valued), and push it to continue.
            raise NotImplementedError("complete the sample branch")
        elif tag == "observe":
            _, a, d, y, m = msg
            # TODO: subtract the log-density of the observed y under d from U (use .sum()),
            # then push y to continue.
            raise NotImplementedError("complete the observe branch")

Model.potential = potential

### Test it

The potential at a chosen trace must equal the model's $-\log\gamma$ computed directly.

In [ ]:
def U_numpy(a, b):
    LOG2PI = math.log(2 * math.pi)
    u = 0.5 * (LOG2PI + a*a) + 0.5 * (LOG2PI + b*b)       # -log N(a;0,1) - log N(b;0,1)
    z = (ys - (a*xs + b)) / SIGMA
    return u + np.sum(0.5 * (LOG2PI + z*z) + math.log(SIGMA))

theta0 = {model.addrs[0]: torch.tensor(1.5), model.addrs[1]: torch.tensor(-0.5)}
U0 = model.potential(theta0).item()
print(f"U(controller) = {U0:.6f}   U(numpy) = {U_numpy(1.5, -0.5):.6f}")
assert np.isclose(U0, U_numpy(1.5, -0.5)), "the sample/observe branches are not right yet"
print("potential OK")

## Use 1 (MAP): inference as optimization (a warm-up)

MAP maximizes $\log\gamma(X)$, i.e. minimizes $U(X)$. The latents are leaf tensors and any
torch optimizer drives them. For this Gaussian model the mode equals the posterior mean, so
we land on the closed form. It is a warm-up: it shows the program runs on supplied latents,
the score is differentiable, and PyTorch can backpropagate through the execution.


In [ ]:
def map_estimate(model, steps=500, lr=0.05):
    theta = {a: torch.zeros((), requires_grad=True) for a in model.addrs}
    opt = torch.optim.Adam(theta.values(), lr=lr)
    for _ in range(steps):
        opt.zero_grad()
        U = model.potential(theta)
        U.backward()
        opt.step()
    return torch.stack([theta[a].detach() for a in model.addrs]).numpy()

map_ab = map_estimate(model)
print(f"MAP (a,b) = ({map_ab[0]:.4f}, {map_ab[1]:.4f})   closed-form ({MODE[0]:.4f}, {MODE[1]:.4f})")
assert np.allclose(map_ab, MODE, atol=1e-2)
print("MAP matches the closed-form mode")

## Use 2 (RWMH): sampling with U alone

A point estimate is not the posterior. The simplest sampler uses **only** `U`, no gradient:
propose an isotropic step $X' = X + \epsilon$, $\epsilon\sim\mathcal{N}(0,\sigma^2 I)$, and accept
with $\alpha=\min\{1,\exp(-U(X')+U(X))\}$. It samples correctly, but this posterior is
**correlated** (a thin diagonal ridge), so isotropic steps fight the geometry and it mixes
slowly. Watch the effective sample size.

In [ ]:
def ess(chain):
    # integrated-autocorrelation effective sample size (Geyer initial-positive truncation)
    chain = np.asarray(chain, float); n = len(chain); c = chain - chain.mean()
    v = np.dot(c, c) / n
    if v == 0: return float(n)
    s = 0.0
    for lag in range(1, n):
        rho = np.dot(c[:-lag], c[lag:]) / (n * v)
        if rho <= 0: break
        s += rho
    return n / (1 + 2 * s)

def rwmh(model, steps, rng, sigma=0.1):
    q = torch.zeros(len(model.addrs))
    U = model.potential({a: q[i] for i, a in enumerate(model.addrs)}).item()
    chain = []; acc = 0
    for _ in range(steps):
        qp = q + torch.as_tensor(rng.normal(0, sigma, size=len(model.addrs)))   # isotropic proposal
        Up = model.potential({a: qp[i] for i, a in enumerate(model.addrs)}).item()
        if math.log(rng.random()) < U - Up:                              # accept on exp(-U' + U)
            q, U = qp, Up; acc += 1
        chain.append(q.numpy().copy())
    return np.array(chain), acc / steps

rw_chain, rw_acc = rwmh(model, 3000, np.random.default_rng(1)); rw = rw_chain[600:]
print(f"RWMH accept {rw_acc:.2f}   ESS(a) {ess(rw[:,0]):.0f}  ESS(b) {ess(rw[:,1]):.0f}"
      f"   mean ({rw[:,0].mean():.3f}, {rw[:,1].mean():.3f})   of {len(rw)}")
assert np.allclose(rw.mean(0), MODE, atol=0.05)
print("RWMH recovers the posterior mean, but with few effective draws")

## Use 3 (HMC): move with the gradient

`torch.autograd.grad` turns the potential into $\nabla U$ with one call. Leapfrog integrates
the trajectory and we accept on the change in the Hamiltonian $H = U + \tfrac12 R^\top R$. The
momentum `R` is drawn fresh each step and discarded; it carries the trajectory **through**
high-density regions instead of halting at the mode. (NumPy RNG for the momentum, so the run
is reproducible.)


In [ ]:
def U_and_grad(self, q):
    q = q.detach().requires_grad_(True)
    U = self.potential({a: q[i] for i, a in enumerate(self.addrs)})
    (g,) = torch.autograd.grad(U, q)
    return U.detach(), g.detach()

Model.U_and_grad = U_and_grad

def hmc(model, steps, rng, T=20, eps=0.06):
    q = torch.zeros(len(model.addrs)); chain = []; acc = 0
    U, grad = model.U_and_grad(q)
    for _ in range(steps):
        p = torch.tensor(rng.normal(size=len(model.addrs)))     # fresh momentum
        H0 = U + 0.5 * p @ p
        q1, p = q.clone(), p - 0.5 * eps * grad
        for _ in range(T):
            q1 = q1 + eps * p
            U1, g1 = model.U_and_grad(q1)
            p = p - eps * g1
        p = p + 0.5 * eps * g1
        H1 = U1 + 0.5 * p @ p
        if math.log(rng.random()) < (H0 - H1).item():
            q, U, grad = q1, U1, g1; acc += 1
        chain.append(q.numpy().copy())
    return np.array(chain), acc / steps

chain, acc = hmc(model, 1500, np.random.default_rng(2)); post = chain[300:]
print(f"HMC accept {acc:.2f}   ESS(a) {ess(post[:,0]):.0f}  ESS(b) {ess(post[:,1]):.0f}"
      f"   mean ({post[:,0].mean():.4f}, {post[:,1].mean():.4f})   closed-form ({MODE[0]:.4f}, {MODE[1]:.4f})")
assert np.allclose(post.mean(0), MODE, atol=0.05)
print("HMC recovers the posterior, with many more effective draws than RWMH")

## Three controllers over one potential

Every method scored with the same `U`; they differ only in whether they use its **gradient**,
and in what they return. RWMH and HMC both sample, but HMC's gradient-guided moves travel
along the ridge, so the same number of steps yields far more effective draws.

In [ ]:
print(f"{'method':5} {'uses U':>7} {'uses grad U':>12}   output")
print(f"{'MAP':5} {'yes':>7} {'yes':>12}   one point   ({map_ab[0]:.3f}, {map_ab[1]:.3f})")
print(f"{'RWMH':5} {'yes':>7} {'no':>12}   samples, ESS(a) {ess(rw[:,0]):.0f} of {len(rw)}")
print(f"{'HMC':5} {'yes':>7} {'yes':>12}   samples, ESS(a) {ess(post[:,0]):.0f} of {len(post)}")

fig, ax = plt.subplots(1, 2, figsize=(9, 4), sharex=True, sharey=True)
for a_, ch, t in zip(ax, [rw, post],
                     [f"RWMH  (ESS a~{ess(rw[:,0]):.0f})", f"HMC  (ESS a~{ess(post[:,0]):.0f})"]):
    a_.plot(ch[:, 0], ch[:, 1], '.', ms=2, alpha=0.3)
    a_.plot(*MODE, 'r+', ms=14, mew=2)                 # the MAP / mode
    a_.set_title(t); a_.set_xlabel("a (slope)")
ax[0].set_ylabel("b (intercept)")
fig.tight_layout()

## Stuck?

The address bookkeeping and the `StaticSupportError` are already written; you only add the
two **scoring** lines. Both subtract a log-density from `U` and differ only in where the
value comes from. `.sum()` collapses a possibly vector-valued `log_prob` to a scalar.

- `sample`: the value is the parameter, `x = theta[a]`. Do `U = U - d.log_prob(x).sum()`, then
  `m.send(x)` so the run continues. Nothing is drawn: the latent is whatever the optimizer
  or the sampler currently holds.
- `observe`: the value `y` is the datum. Do `U = U - d.log_prob(torch.as_tensor(float(y))).sum()`,
  then `m.send(y)`.

`d.log_prob` is differentiable (it is a `torch.distributions.Normal`), so `U` carries a
gradient back to the leaves in `theta`. That is the whole reason MAP and HMC work over the
unchanged evaluator.

## Takeaways

- **Static support is enforced, not assumed.** Pass 1 fixes the latent address set; every
  potential evaluation must visit exactly it, or the controller raises `StaticSupportError`
  instead of returning a wrong density.
- MAP, RWMH, and HMC are **controllers over the Activity 5 message interface**, not new
  evaluators. The runtime is untouched; only `normal` is rebound to a differentiable distribution.
- At `sample`, the controller treats the latent as a **parameter** and scores it into the
  potential `U = -log gamma`. At `observe`, it scores the datum.
- PyTorch makes `U` differentiable, so `U.backward()` gives MAP and `torch.autograd.grad`
  gives the $\nabla U$ that HMC needs.

| method | uses `U`? | uses `grad U`? | output |
|---|:---:|:---:|---|
| MAP  | yes | yes | one point (the mode) |
| RWMH | yes | no  | posterior samples, slow on ridges |
| HMC  | yes | yes | posterior samples, moves with the geometry |

The arc: **static support** fixes a latent vector; **execution** gives $U(X)$; **AD** gives
$\nabla U(X)$; **MAP** optimizes it; **RWMH** samples with $U$ alone; **HMC** uses $\nabla U$ to
sample efficiently.